# Notebook 3: Star Schema Dimensional Modeling

## 1. Overview

This notebook transforms the flat master transaction table (produced in Notebook 1) into a **Star Schema** dimensional model optimized for BI tool ingestion (Power BI / Tableau).

### 1.1. Rationale

The Star Schema design is adopted over the flat master table for three structural reasons:

| Criterion | Flat Table | Star Schema |
|-----------|-----------|-------------|
| Query performance | Full table scans on joins | Minimized joins via surrogate keys |
| Attribute separation | Measures and descriptors mixed | Dimensions (qualitative) separated from Facts (quantitative) |
| BI tool compatibility | Requires manual modeling | Native drag-and-drop model for Power BI / Tableau |

### 1.2. Schema Architecture

The schema comprises one central Fact table and six Dimension tables:

| Table | Type | Description |
|-------|------|-------------|
| `fact_order` | Fact | Transactional measures (`product_cost`, `product_price`) + surrogate keys |
| `dim_customer` | Dimension | Customer demographics, education, and occupation |
| `dim_product` | Dimension | Product catalog with category hierarchy |
| `dim_stores` | Dimension | Physical store geographic reference |
| `dim_date` | Dimension | Calendar dimension (year, quarter, month, day, weekday) |
| `dim_order_status` | Dimension | Order status reference |
| `dim_channel` | Dimension | Sales channel reference |

In [3]:
import pandas as pd
import numpy as np
import os

# Path resolution to load the processed master file
processed_data_path = '../data/processed/master_df.csv'
if not os.path.exists(processed_data_path):
    processed_data_path = 'data/processed/master_df.csv'

# Load the Consolidated Master Data
try:
    df_master = pd.read_csv(processed_data_path, low_memory=False)
    # Convert dates to datetime type
    df_master['order_date'] = pd.to_datetime(df_master['order_date'])
    df_master['dob'] = pd.to_datetime(df_master['dob'])
    print(f"Successfully loaded Consolidated Master Data. Shape: {df_master.shape}")
except FileNotFoundError as e:
    print(f"Error: {e}. Please run previous notebooks first.")


print("\n=== 1. Generating dim_product ===")
product_cols = ['product_code', 'product_profile', 'goods_category', 'category_product_code']
dim_product = df_master[product_cols].drop_duplicates(subset=['product_code']).copy().reset_index(drop=True)
# Generate surrogate key for Product dimension
dim_product.insert(0, 'product_key', range(1, 1 + len(dim_product)))
print(f"dim_product successfully modeled. Shape: {dim_product.shape}")


print("\n=== 2. Generating dim_customer ===")
customer_cols = ['customer_id', 'name_full', 'gender', 'address', 'region', 'dob', 'age', 'education_level', 'job_title']
dim_customer = df_master[customer_cols].drop_duplicates(subset=['customer_id']).copy().reset_index(drop=True)
# Generate surrogate key for Customer dimension
dim_customer.insert(0, 'customer_key', range(1, 1 + len(dim_customer)))
print(f"dim_customer successfully modeled. Shape: {dim_customer.shape}")

Successfully loaded Consolidated Master Data. Shape: (2049263, 24)

=== 1. Generating dim_product ===
dim_product successfully modeled. Shape: (3132, 5)

=== 2. Generating dim_customer ===
dim_customer successfully modeled. Shape: (849472, 10)


## 2. Remaining Dimensions & Fact Table Assembly

With `dim_product` and `dim_customer` established, this section completes the schema by constructing the four remaining dimension tables and assembling the central fact table.

**Key design decisions:**
- `dim_stores` excludes non-physical channel store codes (negative placeholders) to maintain referential integrity of the physical store dimension.
- `dim_date` uses an integer surrogate key (`YYYYMMDD` format) for efficient temporal filtering in BI tools.
- `fact_order` surrogate key mapping uses left joins; online transactions retain a `NULL` `store_key` by design (nullable `Int64` type).
- All tables are exported to `data/processed/` as UTF-8-BOM CSVs for Power BI compatibility.

In [4]:
print("=== 3. Generating dim_stores ===")
# Isolate physical stores (filtering out negative placeholder codes of ecommerce/website)
df_physical_stores = df_master[df_master['store_code'] > 0].copy()
store_cols = ['store_code', 'store_region', 'store_district']
dim_stores = df_physical_stores[store_cols].drop_duplicates(subset=['store_code']).copy().reset_index(drop=True)
dim_stores.insert(0, 'store_key', range(1, 1 + len(dim_stores)))
print(f"dim_stores modeled. Shape: {dim_stores.shape}")


print("\n=== 4. Generating dim_date ===")
# Extract unique order dates across the entire dataset
unique_dates = df_master['order_date'].dt.floor('D').unique()
dim_date = pd.DataFrame({'fully_date': unique_dates}).sort_values(by='fully_date').reset_index(drop=True)
dim_date['date_key'] = dim_date['fully_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['fully_date'].dt.year
dim_date['quarter'] = dim_date['fully_date'].dt.quarter
dim_date['month'] = dim_date['fully_date'].dt.month
dim_date['day'] = dim_date['fully_date'].dt.day
dim_date['weekday'] = dim_date['fully_date'].dt.day_name()
dim_date = dim_date[['date_key', 'fully_date', 'year', 'quarter', 'month', 'day', 'weekday']]
print(f"dim_date modeled. Shape: {dim_date.shape}")


print("\n=== 5. Generating dim_order_status ===")
unique_statuses = df_master['order_status'].unique()
dim_order_status = pd.DataFrame({'status_description': unique_statuses})
dim_order_status.insert(0, 'order_status_key', range(1, 1 + len(dim_order_status)))
print(f"dim_order_status modeled. Shape: {dim_order_status.shape}")


print("\n=== 6. Generating dim_channel ===")
unique_channels = df_master['sales_channel'].unique()
dim_channel = pd.DataFrame({'channel_name': unique_channels})
dim_channel.insert(0, 'channel_key', range(1, 1 + len(dim_channel)))
print(f"dim_channel modeled. Shape: {dim_channel.shape}")


print("\n=== 7. Assembling Central FACT_ORDER Table ===")
df_fact = df_master.copy()

# Crucial fix: Drop pre-existing customer_key from transactional file to prevent merge suffixes (_x, _y)
if 'customer_key' in df_fact.columns:
    df_fact.drop(columns=['customer_key'], inplace=True)

# A. Map CustomerKey
df_fact = pd.merge(df_fact, dim_customer[['customer_id', 'customer_key']], on='customer_id', how='left')

# B. Map ProductKey
df_fact = pd.merge(df_fact, dim_product[['product_code', 'product_key']], on='product_code', how='left')

# C. Map StoreKey (online transactions will gracefully retain a null/NaN store_key)
df_fact = pd.merge(df_fact, dim_stores[['store_code', 'store_key']], on='store_code', how='left')
df_fact['store_key'] = df_fact['store_key'].astype('Int64') # Nullable integer type for online sales

# D. Map DateKey
df_fact['date_key'] = df_fact['order_date'].dt.strftime('%Y%m%d').astype(int)

# E. Map OrderStatusKey
df_fact = pd.merge(df_fact, dim_order_status, left_on='order_status', right_on='status_description', how='left')

# F. Map ChannelKey
df_fact = pd.merge(df_fact, dim_channel, left_on='sales_channel', right_on='channel_name', how='left')

# G. Select final fact columns
fact_columns = [
    'order_id', 'customer_key', 'product_key', 'store_key', 
    'date_key', 'order_status_key', 'channel_key', 'product_cost', 'product_price'
]
fact_order = df_fact[fact_columns].copy()

print(f"fact_order successfully modeled. Shape: {fact_order.shape}")
print(f"Verify fact row count integrity: {len(fact_order) == len(df_master)}")


print("\n=== 8. Exporting Dimensional Schema for BI Ingestion ===")
# Define destination path
processed_dir = '../data/processed/'
if not os.path.exists(processed_dir):
    processed_dir = 'data/processed/'

# Export dimensions and fact table
dim_customer.to_csv(os.path.join(processed_dir, 'dim_customer.csv'), index=False, encoding='utf-8-sig')
dim_product.to_csv(os.path.join(processed_dir, 'dim_product.csv'), index=False, encoding='utf-8-sig')
dim_stores.to_csv(os.path.join(processed_dir, 'dim_stores.csv'), index=False, encoding='utf-8-sig')
dim_date.to_csv(os.path.join(processed_dir, 'dim_date.csv'), index=False, encoding='utf-8-sig')
dim_order_status.to_csv(os.path.join(processed_dir, 'dim_order_status.csv'), index=False, encoding='utf-8-sig')
dim_channel.to_csv(os.path.join(processed_dir, 'dim_channel.csv'), index=False, encoding='utf-8-sig')
fact_order.to_csv(os.path.join(processed_dir, 'fact_order.csv'), index=False, encoding='utf-8-sig')

print(f"Star Schema exported: 6 Dimension tables + 1 Fact table → '{processed_dir}'")

=== 3. Generating dim_stores ===
dim_stores modeled. Shape: (17467, 4)

=== 4. Generating dim_date ===
dim_date modeled. Shape: (1095, 7)

=== 5. Generating dim_order_status ===
dim_order_status modeled. Shape: (5, 2)

=== 6. Generating dim_channel ===
dim_channel modeled. Shape: (4, 2)

=== 7. Assembling Central FACT_ORDER Table ===
fact_order successfully modeled. Shape: (2049263, 9)
Verify fact row count integrity: True

=== 8. Exporting Dimensional Schema for BI Ingestion ===
Star Schema exported: 6 Dimension tables + 1 Fact table → '../data/processed/'
